# Day 2 — YAML Syntax Deep Dive
### `on`, `jobs`, `steps`, `uses`, `with`, `env`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-02-yaml-syntax.ipynb)

**Course:** GitHub Actions for MLOps  
**Day:** 2 of 30  
**Badge level:** Learn

---

In this notebook you will:
- Master the top-level keys of a GitHub Actions workflow file
- Write environment variables at workflow, job, and step scope — and verify precedence
- Use `actions/checkout@v4` and `actions/setup-python@v5` with explicit `with:` inputs
- Write multi-line shell scripts using the YAML `|` literal block scalar
- Read and reference GitHub Actions contexts and expressions

> **Tip:** YAML indentation errors are the number one cause of workflow parse errors. Always use 2-space indentation, never tabs, and validate locally with `actionlint` before pushing.

## Setup

We will use `PyYAML` to parse and validate the YAML strings we write. Run the cell below to install it.

In [ ]:
%pip install pyyaml --quiet

In [ ]:
import yaml

def parse_workflow(raw: str) -> dict:
    '''Parse a YAML workflow string and return the Python dict.'''
    try:
        data = yaml.safe_load(raw)
        print('Parsed successfully!')
        return data
    except yaml.YAMLError as exc:
        print(f'YAML error: {exc}')
        return {}

def show_keys(workflow: dict, path: str = '') -> None:
    '''Recursively print the key structure of a workflow dict.'''
    for k, v in workflow.items():
        full = f'{path}.{k}' if path else k
        print(full)
        if isinstance(v, dict):
            show_keys(v, full)

## 1 — The skeleton of every workflow

Every `.github/workflows/*.yml` file must have three top-level keys:

| Key | Required | Purpose |
|-----|----------|---------|
| `name` | no | Human-readable label shown in the Actions tab |
| `on` | **yes** | Trigger(s) that start the workflow |
| `jobs` | **yes** | Map of one or more jobs to run |

Reference: [Workflow syntax for GitHub Actions](https://docs.github.com/en/actions/writing-workflows/workflow-syntax-for-github-actions)

In [ ]:
skeleton = '''
name: Skeleton workflow

on:
  push:
    branches:
      - main
  pull_request:

jobs:
  hello:
    runs-on: ubuntu-latest
    steps:
      - name: Say hello
        run: echo 'Hello from GitHub Actions'
'''

wf = parse_workflow(skeleton)
print('\nKey structure:')
show_keys(wf)

### Key observations

- `on:` accepts a **map** (with per-trigger config like `branches:`) or a **list** (bare event names).
- `jobs:` is a map — each key is a job ID (letters, digits, `-`, `_`).
- Every job needs `runs-on:` and at least one entry in `steps:`.
- A step must have **either** `run:` (shell command) or `uses:` (action reference) — not both.

## 2 — Environment variable scopes

GitHub Actions lets you declare `env:` at three levels, and **the innermost scope wins** when keys collide:

```
Workflow env  <-- lowest precedence
  Job env     <-- overrides workflow env
    Step env  <-- highest precedence
```

Let's write a workflow that demonstrates all three levels.

In [ ]:
env_scopes_workflow = '''
name: Environment variable scopes

on: [push]

# Workflow-level env -- available in every job and step
env:
  SCOPE: workflow
  GREETING: hello

jobs:
  demonstrate-scopes:
    runs-on: ubuntu-latest

    # Job-level env -- overrides workflow env within this job
    env:
      SCOPE: job
      JOB_ONLY: exists-only-in-job

    steps:
      - name: Print job scope
        run: |
          echo "SCOPE = $SCOPE"          # prints: job
          echo "GREETING = $GREETING"    # prints: hello (inherited)
          echo "JOB_ONLY = $JOB_ONLY"   # prints: exists-only-in-job

      - name: Print step scope
        # Step-level env -- overrides job env within this step only
        env:
          SCOPE: step
          STEP_ONLY: exists-only-in-step
        run: |
          echo "SCOPE = $SCOPE"          # prints: step
          echo "GREETING = $GREETING"    # prints: hello (inherited)
          echo "STEP_ONLY = $STEP_ONLY" # prints: exists-only-in-step

      - name: Back to job scope
        run: |
          echo "SCOPE = $SCOPE"          # prints: job (step env gone)
'''

wf2 = parse_workflow(env_scopes_workflow)

# Verify the three env blocks exist at the right levels
print('Workflow env:', wf2.get('env'))
job = wf2['jobs']['demonstrate-scopes']
print('Job env:', job.get('env'))
step_with_env = job['steps'][1]
print('Step env:', step_with_env.get('env'))

### Precedence rules in plain English

1. A **step** `env:` key shadows the same key set at job or workflow level — **only for that step**.
2. A **job** `env:` key shadows the same key set at workflow level — **for all steps in that job**.
3. Keys set at a higher scope but **not redeclared** at a lower scope are still available (`GREETING` above).
4. Step env is not visible to sibling or later steps — it disappears after the step finishes.

## 3 — Using actions with `uses:` and `with:`

The `uses:` key references a reusable action. The syntax is:

```yaml
uses: owner/repo@ref
```

The `with:` key passes named inputs to the action. Inputs are defined in the action's `action.yml`.

Two actions you will use in almost every Python ML workflow:
- `actions/checkout@v4` — checks out your repository
- `actions/setup-python@v5` — installs a Python version and configures pip caching

In [ ]:
uses_workflow = '''
name: Python setup with actions

on:
  push:
    branches: [main]
  pull_request:

jobs:
  setup:
    runs-on: ubuntu-latest

    steps:
      # Checkout the repository at the commit that triggered the workflow
      - name: Checkout code
        uses: actions/checkout@v4
        with:
          # fetch-depth 0 fetches full history (needed for git log, tags, etc.)
          fetch-depth: 0

      # Install Python and configure the pip cache
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          # cache: pip tells the action to cache ~/.cache/pip between runs
          cache: pip
          # cache-dependency-path pins the cache key to your requirements file
          cache-dependency-path: requirements.txt

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Show Python version
        run: python --version
'''

wf3 = parse_workflow(uses_workflow)

steps = wf3['jobs']['setup']['steps']
for s in steps:
    action = s.get('uses', '(run step)')
    inputs = s.get('with', {})
    print(f"  Step: {s['name']}")
    print(f"    uses: {action}")
    if inputs:
        for k, v in inputs.items():
            print(f"      with.{k}: {v}")

### Why pin to a version tag?

- `actions/checkout@v4` pins to the **major version tag** — you get patch updates automatically but no breaking changes.
- For security-critical workflows you can pin to a **full SHA**: `actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683`.
- Avoid `@main` or `@master` — those can change without notice and break your workflow.

### Common `actions/setup-python@v5` inputs

| Input | Type | Description |
|-------|------|-------------|
| `python-version` | string | e.g. `'3.11'`, `'3.x'`, `'3.11.4'` |
| `cache` | string | `pip`, `pipenv`, or `poetry` |
| `cache-dependency-path` | string | Path to lockfile — changes invalidate cache |

## 4 — Multi-line shell scripts with the `|` block scalar

YAML has two multi-line string styles:

| Style | Key | Newlines | Use case |
|-------|-----|----------|----------|
| Literal | `\|` | Preserved | Shell scripts, prose |
| Folded | `>` | Folded to spaces | Long single-line strings |

For `run:` steps you almost always want `|` — it preserves newlines so each line executes as a separate shell command.

In [ ]:
multiline_workflow = '''
name: Multi-line script demo

on: [push]

jobs:
  scripts:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: System info script
        run: |
          echo === System information ===
          echo "Runner OS: $RUNNER_OS"
          echo "Runner arch: $RUNNER_ARCH"
          echo "Workspace: $GITHUB_WORKSPACE"
          uname -a
          df -h .

      - name: Python environment report
        run: |
          python --version
          python -c 'import sys; print(sys.executable)'
          pip --version

      - name: Folded scalar example (single logical line)
        run: >
          echo this is a very long command that has been
          folded across multiple YAML lines but GitHub
          Actions will run it as one shell command
'''

wf4 = parse_workflow(multiline_workflow)

# Print the run: value for each step to see how PyYAML decoded the scalars
for step in wf4['jobs']['scripts']['steps']:
    if 'run' in step:
        print(f"--- Step: {step['name']} ---")
        print(repr(step['run']))
        print()

### What to notice in the output

- The `|` (literal) scalar preserves every newline — each `echo` is on its own line in the decoded string.
- The `>` (folded) scalar joins the continuation lines with spaces — the whole thing becomes one long line.
- Both scalars strip the **final trailing newline** by default. Append `|+` or `>+` to keep it, or `|-` / `>-` to strip all trailing newlines.

**Rule of thumb:** use `|` for `run:` scripts. Use `>` only when you intentionally want line-folding (e.g. a very long URL).

## 5 — Contexts and expressions

GitHub Actions exposes runtime data through **contexts** — objects you access with `${{ <context>.<property> }}`.

Reference: [Contexts and expressions](https://docs.github.com/en/actions/writing-workflows/choosing-what-your-workflow-does/contexts)

Most-used contexts:

| Context | Example | Contains |
|---------|---------|----------|
| `github` | `${{ github.sha }}` | Event payload, ref, repo, actor, run ID |
| `env` | `${{ env.MY_VAR }}` | Environment variables |
| `secrets` | `${{ secrets.TOKEN }}` | Encrypted secrets |
| `runner` | `${{ runner.os }}` | OS, arch, temp dir |
| `steps` | `${{ steps.my-step.outputs.result }}` | Outputs of previous steps |
| `matrix` | `${{ matrix.python-version }}` | Current matrix combination |

In [ ]:
contexts_workflow = '''
name: Contexts and expressions demo

on:
  push:
    branches: [main]

env:
  MODEL_VERSION: '1.2.0'

jobs:
  context-demo:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - name: Print github context values
        run: |
          echo Commit SHA: ${{ github.sha }}
          echo Branch ref: ${{ github.ref }}
          echo Actor: ${{ github.actor }}
          echo Run ID: ${{ github.run_id }}
          echo Repo: ${{ github.repository }}

      - name: Print env context
        run: |
          echo MODEL_VERSION from env context: ${{ env.MODEL_VERSION }}
          echo MODEL_VERSION from shell: $MODEL_VERSION

      - name: Conditional step using expression
        if: github.ref == 'refs/heads/main'
        run: echo This step only runs on main branch

      - name: Emit a step output
        id: gen-tag
        run: |
          TAG="v${{ env.MODEL_VERSION }}-${{ github.run_number }}"
          echo "tag=$TAG" >> $GITHUB_OUTPUT

      - name: Consume the step output
        run: |
          echo Generated tag: ${{ steps.gen-tag.outputs.tag }}
'''

wf5 = parse_workflow(contexts_workflow)

steps = wf5['jobs']['context-demo']['steps']
print(f'Total steps: {len(steps)}')
for i, s in enumerate(steps):
    cond = f"  [if: {s['if']}]" if 'if' in s else ''
    sid = f" (id={s['id']})" if 'id' in s else ''
    print(f"  {i+1}. {s['name']}{sid}{cond}")

### Key expression patterns

```yaml
# Boolean condition -- runs step only when condition is true
if: github.ref == 'refs/heads/main'

# Negation
if: github.event_name != 'pull_request'

# Logical AND
if: github.ref == 'refs/heads/main' && github.event_name == 'push'

# Functions
if: contains(github.ref, 'release')
if: startsWith(github.ref, 'refs/tags/v')
```

To emit a step output you write `key=value` to the `$GITHUB_OUTPUT` file:
```bash
echo "tag=$TAG" >> $GITHUB_OUTPUT
```
Then reference it in later steps as `${{ steps.<step-id>.outputs.tag }}`.

## 6 — Putting it all together: a complete ML CI workflow

Now let's combine everything — scoped env vars, `uses:` with `with:`, multi-line scripts, and context expressions — into a realistic ML CI workflow.

In [ ]:
complete_workflow = '''
name: ML CI -- lint, test, train smoke

on:
  push:
    branches: [main, release/**]
  pull_request:
    branches: [main]

# Workflow-level defaults
env:
  PYTHON_VERSION: '3.11'
  MODEL_DIR: models
  DATASET_DIR: data

jobs:
  lint:
    name: Lint and type-check
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
          cache-dependency-path: requirements-dev.txt

      - name: Install dev dependencies
        run: pip install -r requirements-dev.txt

      - name: Run ruff
        run: ruff check src/

      - name: Run mypy
        run: mypy src/ --ignore-missing-imports

  test:
    name: Unit tests
    runs-on: ubuntu-latest
    needs: lint

    env:
      # Job-level override -- points tests at the fixture dataset
      DATASET_DIR: tests/fixtures/data

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip
          cache-dependency-path: requirements.txt

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run pytest
        env:
          # Step-level addition -- log level for this run only
          LOG_LEVEL: DEBUG
        run: |
          echo "Running tests against DATASET_DIR=$DATASET_DIR"
          pytest tests/ -v --tb=short

  smoke-train:
    name: Smoke training run
    runs-on: ubuntu-latest
    needs: test
    # Only run full smoke test on pushes to main, not on PRs
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run smoke training
        id: smoke
        run: |
          echo === Starting smoke training run ===
          python src/train.py \
            --epochs 1 \
            --max-samples 100 \
            --output-dir ${{ env.MODEL_DIR }}/smoke
          echo status=passed >> $GITHUB_OUTPUT

      - name: Report result
        run: |
          echo Smoke train status: ${{ steps.smoke.outputs.status }}
          echo Triggered by: ${{ github.actor }} on ${{ github.sha }}
'''

wf6 = parse_workflow(complete_workflow)

print('Jobs defined:')
for job_id, job in wf6['jobs'].items():
    needs = job.get('needs', 'none')
    cond = str(job.get('if', 'always'))[:40]
    n_steps = len(job.get('steps', []))
    print(f'  {job_id}: needs={needs}, if={cond!r}, steps={n_steps}')

### Anatomy of the complete workflow

- **`needs:`** creates a dependency graph — `test` waits for `lint`, `smoke-train` waits for `test`.
- **`if:` on a job** guards the entire job. Here `smoke-train` only runs on pushes to `main`.
- **Three env scopes in one file:**
  - `PYTHON_VERSION` and `MODEL_DIR` set at workflow level.
  - `DATASET_DIR` overridden at job level in `test` to point to fixtures.
  - `LOG_LEVEL` added at step level for the pytest step only.
- The `\\` at the end of a `run:` line is a **shell line continuation** — YAML has already preserved newlines via `|`, so the backslash splits a long shell command, not a YAML string.

## Challenge — Write your own scoped-env workflow

Complete the workflow stub below so that:

1. A workflow-level `env` sets `APP_ENV=production`.
2. A job called `build` overrides it to `APP_ENV=staging`.
3. A step called `Override to dev` inside `build` overrides it further to `APP_ENV=dev` and prints all three variables (`APP_ENV`, `BUILD_ID`, `STEP_SECRET`).
4. `BUILD_ID` is set only at job level.
5. `STEP_SECRET` is set only at step level.
6. A second step called `Back to staging` prints `APP_ENV` without any step-level override — it should show `staging`.

Fill in the `???` placeholders.

In [ ]:
# TODO: replace ??? with correct YAML
challenge_workflow = '''
name: Scoped env challenge

on: [push]

env:
  APP_ENV: ???

jobs:
  build:
    runs-on: ubuntu-latest
    env:
      APP_ENV: ???
      BUILD_ID: ???
    steps:
      - name: Override to dev
        env:
          APP_ENV: ???
          STEP_SECRET: ???
        run: |
          echo "APP_ENV=$APP_ENV"
          echo "BUILD_ID=$BUILD_ID"
          echo "STEP_SECRET=$STEP_SECRET"

      - name: Back to staging
        run: |
          echo "APP_ENV=$APP_ENV"
'''

# Uncomment once you have filled in the ???
# wf_ch = parse_workflow(challenge_workflow)
# job = wf_ch['jobs']['build']
# assert wf_ch['env']['APP_ENV'] == 'production', 'workflow env should be production'
# assert job['env']['APP_ENV'] == 'staging', 'job env should be staging'
# step = job['steps'][0]
# assert step['env']['APP_ENV'] == 'dev', 'step env should be dev'
# print('All assertions passed!')

<details>
<summary>Show solution</summary>

```yaml
name: Scoped env challenge

on: [push]

env:
  APP_ENV: production

jobs:
  build:
    runs-on: ubuntu-latest
    env:
      APP_ENV: staging
      BUILD_ID: build-42
    steps:
      - name: Override to dev
        env:
          APP_ENV: dev
          STEP_SECRET: s3cr3t
        run: |
          echo "APP_ENV=$APP_ENV"
          echo "BUILD_ID=$BUILD_ID"
          echo "STEP_SECRET=$STEP_SECRET"

      - name: Back to staging
        run: |
          echo "APP_ENV=$APP_ENV"
```
</details>

## Day 2 Recap

| Concept | Key point |
|---------|----------|
| Workflow skeleton | `name`, `on`, `jobs` — `on` and `jobs` are required |
| `env:` scopes | workflow < job < step — innermost wins |
| `uses:` | `owner/repo@ref` — always pin to a tag or SHA |
| `with:` | Named inputs to an action — check the action's `action.yml` for available keys |
| `\|` scalar | Preserves newlines — use for `run:` scripts |
| `>` scalar | Folds newlines to spaces — use for long single-line strings |
| Contexts | `${{ github.sha }}`, `${{ env.VAR }}`, `${{ steps.id.outputs.key }}` |
| `if:` | Guards steps or jobs with an expression — evaluates to boolean |
| `$GITHUB_OUTPUT` | Write `key=value` here to emit step outputs |
| `needs:` | Declares job dependencies — creates a DAG |

---

**Next:** Day 3 — Triggers in Depth: `push`, `pull_request`, `schedule`, `workflow_dispatch`, and `workflow_call`